<link href="https://fonts.googleapis.com/css2?family=Oswald:wght@700&display=swap" rel="stylesheet">

<h1 style="
font-family: 'Oswald', sans-serif;
font-weight: 700;
font-style: italic;
font-size: 90px;
letter-spacing: 2px;
color: #E7C173;
text-shadow: 3px 3px 0 #333;
">
MACHINE LEARNING<br>IN INDUSTRY
</h1>

# DIY: Data Preprocessing and Feature Engineering

Practice encoding strategies, missing-value handling, scaling decisions, and feature engineering on **three OpenML datasets**, then add one **local retail-panel exercise** before finishing with a **capstone** where you turn one raw table into a usable feature matrix.

This notebook builds on what you learned in *Day 1 — Data Preprocessing and Feature Engineering* and ends with an end-to-end transfer exercise.

---

### Table of Contents

- [1. Load Datasets](#load)
- [2. Choosing Encodings for High-Cardinality Categories](#ohe-explosion)
- [3. Comparing Encodings on Train and Test](#ohe-overfitting)
- [4. How Should We Encode Zipcode?](#freq-collision)
- [5. Imputing MonthlyIncome](#mean-imputation)
- [6. Missingness and Prediction](#missingness-signal)
- [7. Handling Extreme Values](#clipping-pitfall)
- [8. Scaling Choices Under Outliers](#scaler-choice)
- [9. Dimensionality Reduction Before Prediction](#pca-pitfall)
- [10. Feature Engineering: Grouped Variables for Credit Scoring](#grouped-features)
- [11. Group and Date Features in a Retail Panel](#retail-context)
- [12. Capstone: From Raw Data to a Usable Feature Matrix](#capstone)


---

## Setup

In [ ]:
import os
from pathlib import Path

_root = Path.cwd()
while _root != _root.parent and not (_root / "pyproject.toml").exists():
    _root = _root.parent
os.chdir(_root)
print(f"Working directory: {Path.cwd()}")

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# ── helpers ──
def show(df, n=5):
    display(df.head(n))


def fetch_openml_compat(data_id: int):
    try:
        return fetch_openml(data_id=data_id, as_frame=True, parser="auto")
    except TypeError:
        return fetch_openml(data_id=data_id, as_frame=True)


SEED = 42
np.random.seed(SEED)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

print("Environment ready.")

---

## <a id="load"></a> Section 1 — Load Datasets

We use three datasets from OpenML.

| Dataset | Rows | Features | Task | Description |
|---|---:|---:|---|---|
| **Amazon Employee Access** | ~32 K | 9 categorical | Binary classification (access granted/denied) | [OpenML 4135](https://www.openml.org/d/4135) |
| **King County House Sales** | ~21 K | 19 mixed | Regression (house price) | [OpenML 42092](https://www.openml.org/d/42092) |
| **Give Me Some Credit** | ~150 K | 10 numeric | Binary classification (serious financial distress) | [OpenML 45577](https://www.openml.org/d/45577) |
A later exercise also uses one **local retail panel** (`day1/generated/retail_panel_issues.csv`) because grouped and datetime features are more realistic there than in a purely cross-sectional table.


In [ ]:
# ── Amazon Employee Access (OpenML ID 4135) ──
amazon_raw = fetch_openml_compat(data_id=4135)
amazon = amazon_raw.frame

print("Amazon Employee Access")
print(f"  Shape: {amazon.shape}")
print(f"  Target: '{amazon_raw.target.name}' — positive rate: {amazon_raw.target.astype(int).mean():.4f}")
print()
show(amazon)

In [ ]:
# ── King County House Sales (OpenML ID 42092) ──
kc_raw = fetch_openml_compat(data_id=42092)
kc = kc_raw.frame

print("King County House Sales")
print(f"  Shape: {kc.shape}")
print(f"  Target: '{kc_raw.target.name}' — median: ${kc_raw.target.astype(float).median():,.0f}")
print()
show(kc)

In [ ]:
# ── Give Me Some Credit (OpenML ID 45577) ──
gmsc_raw = fetch_openml_compat(data_id=45577)
gmsc = gmsc_raw.frame

print("Give Me Some Credit")
print(f"  Shape: {gmsc.shape}")
print(f"  Target: '{gmsc_raw.target.name}' — positive rate: {gmsc_raw.target.astype(int).mean():.4f}")
print()
show(gmsc)

In [ ]:
# Your first inspection here


---

## <a id="ohe-explosion"></a> Exercise 2 — First Encoding Decisions (Amazon)

Treat Amazon as a new dataset you have just received.

Inspect the columns, decide what needs encoding, and use `RESOURCE` as one concrete test case.

> **Do It Yourself**
>
> 1. Inspect the Amazon columns and decide which ones need encoding.
> 2. Start with `RESOURCE` and decide which encoding to apply.
> 3. Then encode all the other non-numerical variables.
> 4. Write your own recommendation: when would you still use OHE here, and when would you switch to something else?

######

In [ ]:
# Your exploration here


### Solution / Considerations

One possible path is below.

In [ ]:
# ── One-hot encode only RESOURCE ──
feature_col = "RESOURCE"
ohe_resource = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_resource = ohe_resource.fit_transform(amazon[[feature_col]])

print(f"Original shape using only {feature_col}: {amazon[[feature_col]].shape}")
print(f"OHE shape using only {feature_col}:      {X_resource.shape}")
print(f"Columns / rows:                           {X_resource.shape[1] / X_resource.shape[0]:.2f}")

In [ ]:
# ── OHE on ALL categorical features ──
feature_cols = [c for c in amazon.columns if c != amazon_raw.target.name]

ohe = OneHotEncoder(sparse_output=True, handle_unknown="ignore")
X_ohe = ohe.fit_transform(amazon[feature_cols])

print(f"Original shape:  {amazon[feature_cols].shape}")
print(f"OHE shape:       {X_ohe.shape}")
print(f"Columns / rows:  {X_ohe.shape[1] / X_ohe.shape[0]:.2f}")
print(f"Sparsity:        {1 - X_ohe.nnz / (X_ohe.shape[0] * X_ohe.shape[1]):.6f}")

**Considerations**

- Amazon is almost entirely categorical, so the issue is not **whether** to encode but **how**.
- `RESOURCE` is categorical, but its cardinality is high enough that naive OHE can create an extremely wide matrix.
- OHE is still a strong default for low-cardinality columns.
- As distinct levels become very numerous, more compact encodings such as frequency encoding, target encoding, hashing, or rare-category grouping become easier to justify.

---

### Quick Reference: Models and Metrics Used Below

The remaining exercises use simple models to **measure the impact of preprocessing choices**. You do not need to understand how these models work yet — that is the focus of Day 2. For now, just read the output numbers:

| Concept | What it is | How to read it |
|---|---|---|
| **AUC** (Area Under the ROC Curve) | Measures how well a classifier ranks positive vs negative examples | 0.5 = random guessing, 1.0 = perfect. Higher is better. |
| **R²** (R-squared) | Measures how much variance in the target a regression model explains | 0.0 = no better than predicting the mean, 1.0 = perfect. Can go negative if the model is worse than the mean. |
| **LogisticRegression** | A simple linear classifier | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) |
| **DecisionTreeClassifier** | A single decision tree classifier | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) |
| **KNeighborsClassifier** | A distance-based classifier | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html) |
| **LinearRegression** | A simple linear regression model | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html) |
| **GradientBoostingClassifier** | A tree-based ensemble that builds many small decision trees sequentially | [sklearn docs](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html) |

The key takeaway in every exercise is **relative**: does the metric go up or down when we change the preprocessing? The absolute numbers matter less than the comparison.


---

## <a id="ohe-overfitting"></a> Exercise 3 — Comparing Encodings on Train and Test (Amazon)

Now compare **two models** while changing only how the high-cardinality `RESOURCE` feature is represented.

Brief definition: **overfitting** happens when a model learns patterns that help much more on the training set than on new data. In practice, this often appears as a large train/test performance gap (given the data distribution did not change between the two sets).

Run the cells below and compare what happens for both logistic regression and a decision tree.

> **Do It Yourself**
>
> Compare two models on the same train/test split:
> - `LogisticRegression`
> - `DecisionTreeClassifier`
>
> For each model, compare:
> - naive one-hot encoding (`OneHotEncoder(handle_unknown="ignore")`)
> - one-hot encoding that groups rare levels using `min_frequency` (see the lesson notebook for details on this parameter)
>
> Follow these steps:
>
> **Step 1 — Create two encoders**: one naive, one with `min_frequency` set (remember: when using `min_frequency`, set `handle_unknown="infrequent_if_exist"`)
>
> **Step 2 — Encode**: fit each encoder on `X_train`, transform both `X_train` and `X_test`
>
> **Step 3 — Fit each model and score with `roc_auc_score`** on train and test (use `predict_proba(...)[:, 1]`)
>
> Then answer:
> - How many columns does each encoding produce?
> - Does grouping rare levels reduce the train/test gap for both models?
> - Which model seems more sensitive to rare categories?

In [ ]:
feature_cols = [c for c in amazon.columns if c != amazon_raw.target.name]
y = amazon_raw.target.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    amazon[feature_cols], y, test_size=0.3, random_state=SEED, stratify=y
)

In [ ]:
# Step 1: Create two encoders
# ohe_naive = OneHotEncoder(...)
# ohe_grouped = OneHotEncoder(min_frequency=..., handle_unknown=...)

# Step 2: Fit on X_train, transform both X_train and X_test
# X_tr_naive = ohe_naive.fit_transform(X_train)
# X_te_naive = ohe_naive.transform(X_test)
# ... same for ohe_grouped ...

# Step 3: Fit models and compute roc_auc_score
# lr = LogisticRegression(C=100, max_iter=1000, solver="liblinear", random_state=SEED)
# lr.fit(X_tr_naive, y_train)
# train_auc = roc_auc_score(y_train, lr.predict_proba(X_tr_naive)[:, 1])
# test_auc = roc_auc_score(y_test, lr.predict_proba(X_te_naive)[:, 1])
# ... repeat for grouped, then for DecisionTreeClassifier ...

### Solution / Considerations

One possible path is below.

In [ ]:
# Step 1: Two encoders — naive vs min_frequency
min_count = 20
ohe_naive = OneHotEncoder(handle_unknown="ignore", sparse_output=True)
ohe_grouped = OneHotEncoder(min_frequency=min_count, handle_unknown="infrequent_if_exist", sparse_output=True)

# Step 2: Encode
X_tr_naive = ohe_naive.fit_transform(X_train)
X_te_naive = ohe_naive.transform(X_test)

X_tr_grouped = ohe_grouped.fit_transform(X_train)
X_te_grouped = ohe_grouped.transform(X_test)

print(f"Naive OHE columns:   {X_tr_naive.shape[1]}")
print(f"Grouped OHE columns: {X_tr_grouped.shape[1]}")

# Step 3: Fit and score — Logistic Regression
lr = LogisticRegression(C=100, max_iter=1000, solver="liblinear", random_state=SEED)
lr.fit(X_tr_naive, y_train)
lr_naive_train = roc_auc_score(y_train, lr.predict_proba(X_tr_naive)[:, 1])
lr_naive_test = roc_auc_score(y_test, lr.predict_proba(X_te_naive)[:, 1])

lr2 = LogisticRegression(C=100, max_iter=1000, solver="liblinear", random_state=SEED)
lr2.fit(X_tr_grouped, y_train)
lr_grouped_train = roc_auc_score(y_train, lr2.predict_proba(X_tr_grouped)[:, 1])
lr_grouped_test = roc_auc_score(y_test, lr2.predict_proba(X_te_grouped)[:, 1])

# Step 3: Fit and score — Decision Tree
dt = DecisionTreeClassifier(random_state=SEED)
dt.fit(X_tr_naive, y_train)
dt_naive_train = roc_auc_score(y_train, dt.predict_proba(X_tr_naive)[:, 1])
dt_naive_test = roc_auc_score(y_test, dt.predict_proba(X_te_naive)[:, 1])

dt2 = DecisionTreeClassifier(random_state=SEED)
dt2.fit(X_tr_grouped, y_train)
dt_grouped_train = roc_auc_score(y_train, dt2.predict_proba(X_tr_grouped)[:, 1])
dt_grouped_test = roc_auc_score(y_test, dt2.predict_proba(X_te_grouped)[:, 1])

# Compare
comparison = pd.DataFrame([
    {"Model": "LogReg", "Encoding": "Naive OHE", "Columns": X_tr_naive.shape[1],
     "Train AUC": lr_naive_train, "Test AUC": lr_naive_test, "Gap": lr_naive_train - lr_naive_test},
    {"Model": "LogReg", "Encoding": f"min_frequency={min_count}", "Columns": X_tr_grouped.shape[1],
     "Train AUC": lr_grouped_train, "Test AUC": lr_grouped_test, "Gap": lr_grouped_train - lr_grouped_test},
    {"Model": "DecisionTree", "Encoding": "Naive OHE", "Columns": X_tr_naive.shape[1],
     "Train AUC": dt_naive_train, "Test AUC": dt_naive_test, "Gap": dt_naive_train - dt_naive_test},
    {"Model": "DecisionTree", "Encoding": f"min_frequency={min_count}", "Columns": X_tr_grouped.shape[1],
     "Train AUC": dt_grouped_train, "Test AUC": dt_grouped_test, "Gap": dt_grouped_train - dt_grouped_test},
])
display(comparison)

**Considerations**

- Very rare levels create dummy columns that are active for only a handful of training rows — the model can memorise them.
- `OneHotEncoder`'s `min_frequency` parameter handles this directly: categories below the threshold are merged into a single `infrequent_if_exist` column. You can also use `max_categories` to set a hard cap on the number of output columns.
- The logistic regression is deliberately weakly regularized (`C=100`) and the tree is unconstrained, so the overfitting effect is easier to see.
- Grouping rare levels reduces the train/test gap for both models, but the effect is much stronger for the decision tree.
- No manual preprocessing was needed — the encoder does the grouping internally, which is also safer because it stays consistent between fit and transform.

---

## <a id="freq-collision"></a> Exercise 4 — Encoding `zipcode` (King County)

Inspect the relationship between `zipcode`, how often each value appears, and how house prices vary across zipcodes.

Then decide whether frequency encoding looks acceptable.

> **Do It Yourself**
>
> 1. Inspect the relationship between zipcode frequency and median price.
> 2. Decide whether frequency encoding looks acceptable for `zipcode`.
> 3. Propose at least one alternative encoding that would preserve more information.
>
> You can try one or more of these:
> - **Target encoding**: replace each zipcode with the mean price in the training set.
> - **Ordinal encoding by target**: rank zipcodes by mean price and use the rank.
> - **Binning**: group zipcodes into price tiers and one-hot encode the tiers.
>
> *Hint: compute encodings on the training set only to avoid leakage.*

In [ ]:
show(kc, 2)

In [ ]:
# Your encoding experiment here


### Solution / Considerations

One possible path is below.

In [ ]:
# ── Zipcode summary: frequency vs median price ──
kc["price"] = kc[kc_raw.target.name].astype(float)

zip_stats = (
    kc.groupby("zipcode")
    .agg(count=("price", "size"), median_price=("price", "median"))
    .sort_values("count", ascending=False)
    .reset_index()
)

print(f"Number of zipcodes: {zip_stats.shape[0]}")
print(f"Sales per zipcode — min: {zip_stats['count'].min()}, max: {zip_stats['count'].max()}, median: {zip_stats['count'].median():.0f}")
print()
show(zip_stats, n=10)

In [ ]:
# ── Find collision pairs: similar frequency, very different prices ──
zip_stats_sorted = zip_stats.sort_values("count").reset_index(drop=True)

collisions = []
for i in range(len(zip_stats_sorted)):
    for j in range(i + 1, len(zip_stats_sorted)):
        row_i = zip_stats_sorted.iloc[i]
        row_j = zip_stats_sorted.iloc[j]
        freq_diff = abs(row_i["count"] - row_j["count"])
        price_ratio = max(row_i["median_price"], row_j["median_price"]) / min(row_i["median_price"], row_j["median_price"])
        if freq_diff <= 5 and price_ratio >= 1.8:
            collisions.append({
                "zipcode_A": row_i["zipcode"],
                "count_A": row_i["count"],
                "median_price_A": row_i["median_price"],
                "zipcode_B": row_j["zipcode"],
                "count_B": row_j["count"],
                "median_price_B": row_j["median_price"],
                "price_ratio": price_ratio,
            })

collisions_df = pd.DataFrame(collisions).sort_values("price_ratio", ascending=False)
print(f"Found {len(collisions_df)} zipcode pairs with similar frequency but different prices (ratio >= 1.8x):")
print()
show(collisions_df, n=10)

In [ ]:
# ── Visualize: frequency vs median price scatter ──
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(zip_stats["count"], zip_stats["median_price"] / 1000, alpha=0.7, edgecolors="k", linewidths=0.5)
ax.set_xlabel("Number of sales (= frequency encoding value)")
ax.set_ylabel("Median price ($K)")
ax.set_title("Zipcode frequency vs median price")

# Annotate a collision pair if available
if len(collisions_df) > 0:
    top = collisions_df.iloc[0]
    for label, count_col, price_col in [("A", "count_A", "median_price_A"), ("B", "count_B", "median_price_B")]:
        ax.annotate(
            f"zip {top[f'zipcode_{label}']}",
            xy=(top[count_col], top[price_col] / 1000),
            fontsize=9, fontweight="bold", color="red",
            arrowprops=dict(arrowstyle="->", color="red"),
            xytext=(top[count_col] + 20, top[price_col] / 1000 + 50),
        )

plt.tight_layout()
plt.show()

In [ ]:
# ── After frequency encoding, these zipcodes become indistinguishable ──
if len(collisions_df) > 0:
    top = collisions_df.iloc[0]
    freq_map = kc["zipcode"].value_counts(normalize=True)
    print(f"Zipcode {top['zipcode_A']}:")
    print(f"  Frequency-encoded value: {freq_map[top['zipcode_A']]:.6f}")
    print(f"  Median price: ${top['median_price_A']:,.0f}")
    print()
    print(f"Zipcode {top['zipcode_B']}:")
    print(f"  Frequency-encoded value: {freq_map[top['zipcode_B']]:.6f}")
    print(f"  Median price: ${top['median_price_B']:,.0f}")
    print()
    print(f"Frequency difference: {abs(freq_map[top['zipcode_A']] - freq_map[top['zipcode_B']]):.6f}")
    print(f"Price ratio: {top['price_ratio']:.1f}x")
    print()
    print("A model using frequency encoding treats these two neighborhoods as nearly identical.")

**Considerations**

- Categories with similar frequencies can map to the same or nearly the same encoded value even when their relationship to the target is very different.
- Frequency encoding keeps the matrix compact, but it can also erase meaningful category differences.

---

## <a id="mean-imputation"></a> Exercise 5 — Imputing `MonthlyIncome` (Give Me Some Credit)

Start by looking at the observed distribution before deciding on a fill rule.

The main question is whether one global number is a reasonable stand-in for every missing income value.

> **Do It Yourself**
>
> Try these alternatives to global mean imputation and compare the resulting distributions:
> - **Median imputation** — does it reduce the distortion compared to the mean?
> - **Group-conditional imputation** — impute with the median of the borrower's age group (e.g., 10-year bins). How does the filled distribution compare?
> - **Random-sample imputation** — for each missing value, sample randomly from observed values. Does this preserve the distribution shape better?
>
> *Think about: which method would you choose if your downstream model is a logistic regression? What if it is a gradient-boosted tree?*

In [ ]:
show(gmsc)

In [ ]:
# Your imputation experiment here


### Solution / Considerations

One possible path is below.

In [ ]:
# ── MonthlyIncome distribution ──
mi = gmsc["MonthlyIncome"]
mi_observed = mi.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw distribution with mean/median lines
axes[0].hist(mi_observed.clip(upper=30_000), bins=60, edgecolor="k", linewidth=0.3, alpha=0.8)
axes[0].axvline(mi_observed.mean(), color="red", ls="--", lw=2, label=f"Mean: ${mi_observed.mean():,.0f}")
axes[0].axvline(mi_observed.median(), color="blue", ls="--", lw=2, label=f"Median: ${mi_observed.median():,.0f}")
axes[0].set_xlabel("MonthlyIncome (clipped at $30K for visibility)")
axes[0].set_ylabel("Count")
axes[0].set_title("Observed distribution")
axes[0].legend()

# Right: after mean imputation — spike at the mean
mi_mean_filled = mi.fillna(mi_observed.mean())
axes[1].hist(mi_mean_filled.clip(upper=30_000), bins=60, edgecolor="k", linewidth=0.3, alpha=0.8)
axes[1].axvline(mi_observed.mean(), color="red", ls="--", lw=2, label=f"Mean: ${mi_observed.mean():,.0f}")
axes[1].set_xlabel("MonthlyIncome (clipped at $30K for visibility)")
axes[1].set_title(f"After mean imputation ({mi.isnull().sum():,} values filled)")
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Values exactly at the mean after imputation: {(mi_mean_filled == mi_observed.mean()).sum():,} ({(mi_mean_filled == mi_observed.mean()).mean():.1%} of all rows)")
print(f"Standard deviation — before: {mi_observed.std():,.0f} | after: {mi_mean_filled.std():,.0f} (shrunk by {1 - mi_mean_filled.std()/mi_observed.std():.1%})")

In [ ]:
# ── Subgroup structure: MonthlyIncome by age group ──
age = gmsc["age"]
mi_obs_mask = mi.notna()

age_bins = [(21, 35, "21-35"), (35, 50, "35-50"), (50, 65, "50-65"), (65, 85, "65-85")]

fig, ax = plt.subplots(figsize=(12, 5))
for lo, hi, label in age_bins:
    mask = mi_obs_mask & (age >= lo) & (age < hi)
    subset = mi[mask].clip(upper=25_000)
    ax.hist(subset, bins=50, alpha=0.45, label=f"Age {label} (median=${mi[mask].median():,.0f})")

global_mean = mi_observed.mean()
ax.axvline(global_mean, color="red", ls="--", lw=2, label=f"Global mean: ${global_mean:,.0f}")
ax.set_xlabel("MonthlyIncome (clipped at $25K)")
ax.set_ylabel("Count")
ax.set_title("Income distribution by age group — the global mean misrepresents every subgroup")
ax.legend()
plt.tight_layout()
plt.show()

print("If a 25-year-old has missing income, imputing $6,670 (the global mean) overshoots their likely income by ~$2,600.")
print("If a 55-year-old has missing income, it undershoots by ~$1,000.")

**Considerations**

- A single global mean can distort a skewed or multi-population distribution.
- Median or group-aware imputation often preserves the structure better, and the best choice depends on the downstream model and the goal of the analysis.

---

## <a id="missingness-signal"></a> Exercise 6 — Missingness and Prediction (Give Me Some Credit)

Before treating missing values as a nuisance, check whether the missingness pattern itself is informative.

The question here is: do rows with missing values behave differently from rows with observed values?

> **Do It Yourself**
>
> Explore further:
> - Add a **missing indicator** column for each feature with missing values. Train a `LogisticRegression` with and without the indicators — does test AUC improve?
> - Try `HistGradientBoostingClassifier`, which handles missing values natively (no imputation needed). Compare its AUC to the logistic regression with median imputation.
> - Does the missing indicator help more for `MonthlyIncome` or `NumberOfDependents`? Try adding them one at a time.
>
> *Think about: in a production pipeline, would you always add missing indicators, or only when you have evidence that missingness is informative?*

In [ ]:
show(gmsc, 2)

In [ ]:
# Your missingness experiment here


### Solution / Considerations

One possible path is below.

In [ ]:
# ── Default rate by missingness pattern ──
target = gmsc_raw.target.astype(int)

miss_cols = gmsc.columns[gmsc.isnull().any()].tolist()
print("Default rate by missingness:\n")
for col in miss_cols:
    is_missing = gmsc[col].isnull()
    rate_miss = target[is_missing].mean()
    rate_pres = target[~is_missing].mean()
    n_miss = is_missing.sum()
    print(f"  {col}")
    print(f"    Missing ({n_miss:,} rows):  default rate = {rate_miss:.4f}")
    print(f"    Present ({(~is_missing).sum():,} rows): default rate = {rate_pres:.4f}")
    print(f"    Ratio: {rate_miss / rate_pres:.2f}x")
    print()

In [ ]:
# ── Visualize: target distribution split by missingness ──
fig, axes = plt.subplots(1, len(miss_cols), figsize=(7 * len(miss_cols), 5))
if len(miss_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, miss_cols):
    is_missing = gmsc[col].isnull()
    
    rates = pd.DataFrame({
        "Group": [f"{col}\nmissing", f"{col}\npresent", "Overall"],
        "Default rate": [target[is_missing].mean(), target[~is_missing].mean(), target.mean()],
    })
    
    colors = ["#c0392b", "#2980b9", "#7f8c8d"]
    ax.bar(rates["Group"], rates["Default rate"], color=colors, edgecolor="k", linewidth=0.5)
    ax.set_ylabel("Default rate")
    ax.set_title(f"Default rate by {col} missingness")
    
    for i, row in rates.iterrows():
        ax.text(i, row["Default rate"] + 0.002, f"{row['Default rate']:.4f}", ha="center", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ── What happens if you drop rows with missing values? ──
complete_mask = gmsc.notna().all(axis=1)
n_total = len(gmsc)
n_complete = complete_mask.sum()

print(f"Total rows:    {n_total:,}")
print(f"Complete rows: {n_complete:,} ({n_complete/n_total:.1%})")
print(f"Dropped:       {n_total - n_complete:,} ({(n_total - n_complete)/n_total:.1%})")
print()
print(f"Default rate — all rows:      {target.mean():.4f}")
print(f"Default rate — complete only:  {target[complete_mask].mean():.4f}")
print(f"Default rate — dropped rows:   {target[~complete_mask].mean():.4f}")
print()
print("Dropping rows removes the lower-risk subpopulation and shifts the class balance.")
print("This is evidence of MAR/MNAR — the missingness is not random with respect to the target.")
print()

# ── Age distribution shift ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(gmsc.loc[complete_mask, "age"], bins=40, alpha=0.7, label="Complete cases", density=True)
axes[0].hist(gmsc.loc[~complete_mask, "age"], bins=40, alpha=0.7, label="Dropped rows", density=True)
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Density")
axes[0].set_title("Age distribution: complete vs dropped rows")
axes[0].legend()

axes[1].hist(gmsc.loc[complete_mask, "MonthlyIncome"].clip(upper=20_000), bins=40, alpha=0.7, label="Complete cases", density=True)
mi_dropped = gmsc.loc[~complete_mask & gmsc["MonthlyIncome"].notna(), "MonthlyIncome"]
if len(mi_dropped) > 0:
    axes[1].hist(mi_dropped.clip(upper=20_000), bins=40, alpha=0.7, label="Dropped (income present, deps missing)", density=True)
axes[1].set_xlabel("MonthlyIncome (clipped at $20K)")
axes[1].set_ylabel("Density")
axes[1].set_title("Income distribution shift from dropping rows")
axes[1].legend()

plt.tight_layout()
plt.show()

**Considerations**

- Missingness can carry signal.
- If rows with missing values have systematically different target rates, a missing-indicator feature may be useful rather than redundant.

---

## <a id="clipping-pitfall"></a> Exercise 7 — Handling Extreme Values (Give Me Some Credit)

Extreme values often deserve attention, but not every extreme should be clipped away.

Start by inspecting how default risk changes across the range of `RevolvingUtilizationOfUnsecuredLines`. Then ask: if we cap everything above `1.0`, what information disappears?

> **Do It Yourself**
>
> 1. Summarize default rate for `RevolvingUtilizationOfUnsecuredLines` across a few utilization bands.
> 2. Compare the bands just below and above `1.0`.
> 3. Decide whether clipping at `1.0` would erase a meaningful risk regime.
> 4. If you want, suggest a safer alternative than blind clipping.
>
> *Hint: focus especially on the difference between `0.7–1.0` and `1.0–2.0`.*

In [ ]:
# Your clipping alternative experiment here


### Solution / Considerations

One possible path is below.

In [ ]:
# ── Default rate by RevolvingUtilization band ──
rev_col = "RevolvingUtilizationOfUnsecuredLines"
rev = gmsc[rev_col].astype(float)
target = gmsc_raw.target.astype(int)

# Show how extreme the outliers are
print(f"{rev_col}")
print(f"  min: {rev.min():.4f}  |  median: {rev.median():.4f}  |  99th pct: {rev.quantile(0.99):.4f}  |  max: {rev.max():,.0f}")
print(f"  Values > 1.0:  {(rev > 1).sum():,} rows ({(rev > 1).mean():.2%})")
print(f"  Values > 10:   {(rev > 10).sum():,} rows")
print()

# Default rate by utilization band
bands = [
    (0.0, 0.3, "0–0.3 (healthy)"),
    (0.3, 0.7, "0.3–0.7 (moderate)"),
    (0.7, 1.0, "0.7–1.0 (near limit)"),
    (1.0, 2.0, "1.0–2.0 (over limit)"),
    (2.0, float("inf"), ">2.0 (extreme)"),
]

print("Default rate by utilization band:\n")
band_data = []
for lo, hi, label in bands:
    mask = (rev >= lo) & (rev < hi)
    n = mask.sum()
    rate = target[mask].mean() if n > 0 else 0
    band_data.append({"band": label, "n_rows": n, "default_rate": rate})
    print(f"  {label:25s}  n={n:>6,}  default rate = {rate:.4f}")

print()
print("Inspect the differences across bands before deciding whether clipping is appropriate.")

In [ ]:
# ── Visualize: default rate by utilization band ──
band_df = pd.DataFrame(band_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: default rate by band
colors = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]
axes[0].bar(band_df["band"], band_df["default_rate"], color=colors, edgecolor="k", linewidth=0.5)
axes[0].set_ylabel("Default rate")
axes[0].set_title("Default rate by utilization band")
axes[0].tick_params(axis="x", rotation=25)
for i, row in band_df.iterrows():
    axes[0].text(i, row["default_rate"] + 0.005, f"{row['default_rate']:.3f}", ha="center", fontsize=9)

# Right: distribution of RevolvingUtilization colored by target
rev_default = rev[target == 1]
rev_no_default = rev[target == 0]
axes[1].hist(rev_no_default.clip(upper=5), bins=80, alpha=0.6, label="No default", density=True)
axes[1].hist(rev_default.clip(upper=5), bins=80, alpha=0.6, label="Default", density=True)
axes[1].axvline(1.0, color="red", ls="--", lw=2, label="Clip threshold (1.0)")
axes[1].set_xlabel(f"{rev_col} (clipped at 5 for visibility)")
axes[1].set_ylabel("Density")
axes[1].set_title("Utilization distribution by target")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── What clipping at 1.0 erases ──
tail_summary = pd.DataFrame([
    {
        "Band": "0.7–1.0 (near limit)",
        "Rows": ((rev >= 0.7) & (rev < 1.0)).sum(),
        "Default rate": target[(rev >= 0.7) & (rev < 1.0)].mean(),
        "Value after clip at 1.0": "unchanged (< 1.0)",
    },
    {
        "Band": "1.0–2.0 (over limit)",
        "Rows": ((rev >= 1.0) & (rev < 2.0)).sum(),
        "Default rate": target[(rev >= 1.0) & (rev < 2.0)].mean(),
        "Value after clip at 1.0": "1.0",
    },
    {
        "Band": ">2.0 (extreme tail)",
        "Rows": (rev >= 2.0).sum(),
        "Default rate": target[rev >= 2.0].mean(),
        "Value after clip at 1.0": "1.0",
    },
])
display(tail_summary)
print()
print(f"Rows with utilization > 1.0 that become exactly 1.0 after clipping: {(rev > 1.0).sum():,}")
print("Clipping at 1.0 collapses the entire over-limit tail into a single value.")
print("That may be acceptable if those rows are pure noise, but the band summary above suggests they are a distinct risk regime.")
print()
# ── Prepare a clean train/test split for the next exercise ──
gmsc_features = [c for c in gmsc.columns if c != gmsc_raw.target.name]
complete = gmsc[gmsc_features].dropna().astype(float)
y_complete = target.loc[complete.index]

X_tr, X_te, y_tr, y_te = train_test_split(
    complete, y_complete, test_size=0.3, random_state=SEED, stratify=y_complete
)
print(f"Prepared complete-case split for Exercise 10: {X_tr.shape[0]:,} train rows / {X_te.shape[0]:,} test rows")


**Considerations**

- The jump from `0.7–1.0` to `1.0–2.0` suggests that crossing the credit limit marks a meaningfully different risk regime.
- Blind clipping at `1.0` removes that distinction by forcing every over-limit borrower onto the same capped value.
- A safer alternative is often to keep the raw value, transform it, or add an explicit `OverLimitFlag` rather than flattening the tail immediately.
- If the extreme tail is sparse, be careful not to over-interpret it. The cleaner lesson here is that clipping can erase signal, not that every large value must be preserved unchanged.

---

## <a id="scaler-choice"></a> Exercise 8 — Scaling Choices Under Outliers

Scaling is not just about putting variables on the same range. With distance-based models, the **choice of scaler** can change how much outliers distort the geometry of the problem.

Here you will compare four versions of the same `KNeighborsClassifier` on an outlier-heavy subset of **Give Me Some Credit**:
- no scaling
- `StandardScaler`
- `MinMaxScaler`
- `RobustScaler`

The goal is to see whether a scaler that is less sensitive to extreme values gives better generalization.


> **Do It Yourself**
>
> 1. Take a manageable sample of the data and keep a few numeric columns with heavy tails.
> 2. Train the same `KNeighborsClassifier` four times: no scaling, standard, min-max, robust.
> 3. Compare **train AUC**, **test AUC**, and the **train-test gap**.
> 4. Decide whether one scaler clearly handles the outliers better.
>
> Follow these steps:
>
> **Step 1 — Sample and split**: take ~8k rows, pick the outlier-heavy columns (`RevolvingUtilizationOfUnsecuredLines`, `DebtRatio`, `MonthlyIncome`, plus a couple of cleaner ones), and do a `train_test_split`.
>
> **Step 2 — Build a Pipeline for each scaler**: use the same pattern from the lesson notebook — `Pipeline([("imputer", SimpleImputer(...)), ("scaler", ...), ("model", KNeighborsClassifier(...))])`. Run it four times swapping only the scaler step (or skipping it for the "no scaling" case).
>
> **Step 3 — Score each**: `roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])` — same pattern as earlier exercises.
>
> *Hint: `RevolvingUtilizationOfUnsecuredLines`, `DebtRatio`, and `MonthlyIncome` are the columns to watch.*

In [ ]:
# Step 1: Sample and split
# scale_sample = gmsc.sample(n=8_000, random_state=SEED).copy()
# scale_features = ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio", "MonthlyIncome", "age", "NumberOfOpenCreditLinesAndLoans"]
# X_scale_train, X_scale_test, y_scale_train, y_scale_test = train_test_split(...)

# Step 2: Build a Pipeline for each scaler (same pattern as the lesson notebook)
# Example for one scaler:
# pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", RobustScaler()),
#     ("model", KNeighborsClassifier(n_neighbors=25)),
# ])
# pipe.fit(X_scale_train, y_scale_train)

# Step 3: Score
# train_auc = roc_auc_score(y_scale_train, pipe.predict_proba(X_scale_train)[:, 1])
# test_auc = roc_auc_score(y_scale_test, pipe.predict_proba(X_scale_test)[:, 1])

# Repeat for: no scaler, StandardScaler, MinMaxScaler, RobustScaler
# Collect results into a DataFrame and display

### Solution / Considerations

One possible path is below.


In [ ]:
# ── Step 1: Sample and split ──
scale_sample = gmsc.sample(n=8_000, random_state=SEED).copy()
scale_target = scale_sample[gmsc_raw.target.name].astype(int)
scale_features = [
    "RevolvingUtilizationOfUnsecuredLines",
    "DebtRatio",
    "MonthlyIncome",
    "age",
    "NumberOfOpenCreditLinesAndLoans",
]

X_scale_train, X_scale_test, y_scale_train, y_scale_test = train_test_split(
    scale_sample[scale_features],
    scale_target,
    test_size=0.3,
    random_state=SEED,
    stratify=scale_target,
)

# ── Step 2 + 3: Pipeline per scaler, fit and score ──
results = []
scalers = [
    ("No scaling", None),
    ("StandardScaler", StandardScaler()),
    ("MinMaxScaler", MinMaxScaler()),
    ("RobustScaler", RobustScaler()),
]

for name, scaler in scalers:
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scaler is not None:
        steps.append(("scaler", scaler))
    steps.append(("model", KNeighborsClassifier(n_neighbors=25)))

    pipe = Pipeline(steps)
    pipe.fit(X_scale_train, y_scale_train)

    train_auc = roc_auc_score(y_scale_train, pipe.predict_proba(X_scale_train)[:, 1])
    test_auc = roc_auc_score(y_scale_test, pipe.predict_proba(X_scale_test)[:, 1])
    results.append({"Scaler": name, "Train AUC": train_auc, "Test AUC": test_auc, "Gap": train_auc - test_auc})

scaler_comparison = pd.DataFrame(results).sort_values("Test AUC", ascending=False)
scaler_comparison["AUC gain vs no scaling"] = (
    scaler_comparison["Test AUC"] - scaler_comparison.loc[scaler_comparison["Scaler"] == "No scaling", "Test AUC"].iloc[0]
)
display(scaler_comparison)

**Considerations**

- On this outlier-heavy subset, `RobustScaler` usually gives the highest **test AUC** and the smallest **train-test gap**.
- `MinMaxScaler` can still struggle because one extreme maximum or minimum can squeeze most rows into a narrow range.
- This is why scaling is not only about `yes/no`, but also about **which scaler** matches the feature distribution and the model family.
- We use `KNeighborsClassifier` here because distance-based models make the scaling effect visible very quickly.


---

## <a id="pca-pitfall"></a> Exercise 9 — Dimensionality Reduction Before Prediction

Before treating PCA as a harmless preprocessing step, check whether the retained high-variance directions are actually the predictive ones.

The key question is not just whether PCA compresses the data, but whether it preserves the information needed for prediction.

We demonstrate this with a small synthetic 2D dataset where the target lives mostly in a low-variance direction.

> **Do It Yourself**
>
> 1. Build a small comparison between PCA and a no-PCA baseline.
> 2. Inspect both the scores and the plot.
> 3. Write one short conclusion: when might unsupervised dimensionality reduction remove useful signal?

In [ ]:
# Your PCA experiment here


### Solution / Considerations

One possible path is below.

In [ ]:
# ── Build a 2D dataset where the target lives in the low-variance direction ──
from sklearn.linear_model import LinearRegression
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

rng = np.random.RandomState(SEED)
n = 800
u = rng.normal(0, 3.0, size=n)   # high-variance latent direction
v = rng.normal(0, 0.25, size=n)  # low-variance latent direction

x1 = u + v
x2 = u - v
X_synth = np.column_stack([x1, x2])
y_synth = 6 * v + rng.normal(0, 0.15, size=n)

X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_synth, y_synth, random_state=SEED)

pcr = make_pipeline(StandardScaler(), PCA(n_components=1), LinearRegression())
lr = make_pipeline(StandardScaler(), LinearRegression())

pcr.fit(X_tr_s, y_tr_s)
lr.fit(X_tr_s, y_tr_s)

r2_pcr = pcr.score(X_te_s, y_te_s)
r2_lr = lr.score(X_te_s, y_te_s)

comp = pd.DataFrame({
    "Method": ["PCA(1) + LinearRegression", "LinearRegression (no PCA)"],
    "R²": [r2_pcr, r2_lr],
})
display(comp)
print()
print(f"With PCA(1):    R² = {r2_pcr:.3f}")
print(f"Without PCA:   R² = {r2_lr:.3f}")


In [ ]:
# ── Why PCA fails here ──
X_centered = StandardScaler().fit_transform(X_tr_s)
pca_full = PCA(n_components=2).fit(X_centered)
explained = pca_full.explained_variance_ratio_

print(f"PC1 explained variance ratio: {explained[0]:.3f}")
print(f"PC2 explained variance ratio: {explained[1]:.3f}")
print()
print("The target was constructed from the low-variance direction (roughly x1 - x2).")
print("Keeping only PC1 preserves most of the variance but throws away most of the predictive signal.")


In [ ]:
# ── Visualize: predictions with and without PCA ──
y_pred_pcr = pcr.predict(X_te_s)
y_pred_lr = lr.predict(X_te_s)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(y_te_s, y_pred_pcr, alpha=0.3)
axes[0].plot([y_te_s.min(), y_te_s.max()], [y_te_s.min(), y_te_s.max()], "r--", lw=2)
axes[0].set_xlabel("True y")
axes[0].set_ylabel("Predicted y")
axes[0].set_title(f"With PCA(1): R² = {r2_pcr:.3f}")

axes[1].scatter(y_te_s, y_pred_lr, alpha=0.3)
axes[1].plot([y_te_s.min(), y_te_s.max()], [y_te_s.min(), y_te_s.max()], "r--", lw=2)
axes[1].set_xlabel("True y")
axes[1].set_ylabel("Predicted y")
axes[1].set_title(f"Without PCA: R² = {r2_lr:.3f}")

plt.tight_layout()
plt.show()

print("Inspect the two plots and decide what information PCA preserved or removed.")


**Considerations**

- PCA is unsupervised: it preserves variance, not predictive signal.
- If the target aligns with a low-variance direction, PCA can remove exactly what the model needs.

---

## <a id="grouped-features"></a> Exercise 10 — Feature Engineering: Grouped Variables for Credit Scoring

Raw features rarely tell the full story. In credit scoring, **ratios and interactions** between variables often carry more signal than individual columns.

For example, knowing someone's `DebtRatio` and `MonthlyIncome` separately is useful — but their product gives you `MonthlyDebt`, which directly measures the dollar burden on the borrower.

The Give Me Some Credit dataset has 10 features describing a borrower's financial profile. Your task is to **create new grouped/ratio features** and check whether they help a simple linear model.

Here are some ideas from top Kaggle solutions:

| New feature | Formula | Intuition |
|---|---|---|
| `MonthlyDebt` | `DebtRatio × MonthlyIncome` | Dollar amount of monthly debt obligations |
| `IncomePerDependent` | `MonthlyIncome / (NumberOfDependents + 1)` | How stretched is the household income? |
| `TotalLatePay` | `Num30-59 + Num60-89 + Num90+` | Aggregate delinquency count across all severity levels |
| `OverLimitFlag` | `RevolvingUtilization > 1.0` | Binary: is the borrower over their credit limit? |
| `HighDebtFlag` | `DebtRatio > 1.0` | Binary: do debts exceed income? |
| `LatePay_per_Line` | `TotalLatePay / (NumOpenLines + 1)` | Rate of delinquency relative to number of open accounts |

📚 [NYC Data Science Blog — Give Me Some Credit](https://nycdatascience.com/blog/student-works/kaggle-predict-consumer-credit-default/) · [Kaggle: Give Me Some Credit](https://www.kaggle.com/c/GiveMeSomeCredit)

> **Do It Yourself**
>
> Create at least **3 new grouped/ratio features** from the table above (or invent your own) and measure whether they improve a `LogisticRegression`.
>
> Follow these steps:
>
> **Step 1 — Baseline**: build a `Pipeline` with `SimpleImputer(strategy="median")` → `StandardScaler()` → `LogisticRegression()`. Fit on `X_tr`, score on `X_te` with `roc_auc_score`. This is your baseline AUC.
>
> **Step 2 — Engineer features**: add new columns to copies of `X_tr` and `X_te`. Some ideas from the table:
> - `TotalLatePay = NumberOfTime30-59DaysPastDueNotWorse + NumberOfTime60-89DaysPastDueNotWorse + NumberOfTimes90DaysLate`
> - `MonthlyDebt = DebtRatio * MonthlyIncome`
> - `IncomePerDependent = MonthlyIncome / (NumberOfDependents + 1)`
>
> **Step 3 — Fit the same pipeline** on the enriched data and compare test AUC to baseline.
>
> **Step 4 — Inspect coefficients**: look at `pipe.named_steps["logisticregression"].coef_` to see if engineered features rank in the top 10 by absolute magnitude.
>
> *Think about: linear models cannot invent arbitrary ratios or thresholds by themselves. Does feature engineering help more here than it would for a tree model?*

In [ ]:
# Step 1: Baseline pipeline (same pattern as the lesson notebook)
# baseline_pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("scaler", StandardScaler()),
#     ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
# ])
# baseline_pipe.fit(X_tr, y_tr)
# auc_baseline = roc_auc_score(y_te, baseline_pipe.predict_proba(X_te)[:, 1])

# Step 2: Add engineered features
# X_tr_eng = X_tr.copy()
# X_te_eng = X_te.copy()
# for df in [X_tr_eng, X_te_eng]:
#     df["TotalLatePay"] = df["NumberOfTime30-59DaysPastDueNotWorse"] + ...
#     df["MonthlyDebt"] = df["DebtRatio"] * df["MonthlyIncome"]
#     df["IncomePerDependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)

# Step 3: Fit same pipeline on enriched data
# eng_pipe = Pipeline([...])  # same as baseline
# eng_pipe.fit(X_tr_eng, y_tr)
# auc_eng = roc_auc_score(y_te, eng_pipe.predict_proba(X_te_eng)[:, 1])

# Step 4: Inspect coefficients
# coefs = pd.Series(
#     np.abs(eng_pipe.named_steps["model"].coef_[0]),
#     index=X_tr_eng.columns,
# ).sort_values(ascending=False)
# display(coefs.head(10))

### Solution / Considerations

One possible path is below.

In [ ]:
# ── Step 1: Baseline ──
baseline_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])
baseline_pipe.fit(X_tr, y_tr)
auc_baseline = roc_auc_score(y_te, baseline_pipe.predict_proba(X_te)[:, 1])
print(f"Baseline Test AUC: {auc_baseline:.4f}")

# ── Step 2: Engineer features ──
X_tr_eng = X_tr.copy()
X_te_eng = X_te.copy()

for df in [X_tr_eng, X_te_eng]:
    df["TotalLatePay"] = (
        df["NumberOfTime30-59DaysPastDueNotWorse"]
        + df["NumberOfTime60-89DaysPastDueNotWorse"]
        + df["NumberOfTimes90DaysLate"]
    )
    df["MonthlyDebt"] = df["DebtRatio"] * df["MonthlyIncome"]
    df["IncomePerDependent"] = df["MonthlyIncome"] / (df["NumberOfDependents"] + 1)
    df["OverLimitFlag"] = (df["RevolvingUtilizationOfUnsecuredLines"] > 1.0).astype(int)
    df["LatePay_per_Line"] = df["TotalLatePay"] / (df["NumberOfOpenCreditLinesAndLoans"] + 1)

# ── Step 3: Fit same pipeline on enriched data ──
eng_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, random_state=SEED)),
])
eng_pipe.fit(X_tr_eng, y_tr)
auc_eng = roc_auc_score(y_te, eng_pipe.predict_proba(X_te_eng)[:, 1])

comparison = pd.DataFrame({
    "Model": ["Baseline", "Baseline + engineered features"],
    "Test AUC": [auc_baseline, auc_eng],
    "AUC gain": [0.0, auc_eng - auc_baseline],
})
display(comparison)

# ── Step 4: Inspect coefficients ──
coefs = pd.Series(
    np.abs(eng_pipe.named_steps["model"].coef_[0]),
    index=X_tr_eng.columns,
).sort_values(ascending=False)

print("\nTop 10 coefficients by absolute magnitude:")
display(coefs.head(10).to_frame("abs_coefficient"))

**Considerations**

- For a linear model, engineered ratios and flags can create structure the model could not express easily from the raw columns alone.
- In this dataset, grouped features such as `TotalLatePay`, `LatePay_per_Line`, and `OverLimitFlag` often carry strong predictive signal.
- Tree models can discover some interactions on their own, so the gain from explicit feature engineering is often smaller there.
- Good engineered features should reflect a relationship you can explain, not just an arbitrary arithmetic combination.

---

## <a id="retail-context"></a> Exercise 11 — Group and Date Features in a Retail Panel

A raw row often misses context. In panel data, the same store can behave differently across time, and a raw numeric identifier such as `store_id` is usually **not** the feature you really want.

In this exercise, compare four versions of the same `LinearRegression` on a local retail panel:
- base operational features only
- base features + raw `store_id`
- base features + a **train-only** store-level aggregate
- base features + the store aggregate + simple date-derived features

The goal is to see whether contextual features help more than the raw identifier itself.


> **Do It Yourself**
>
> 1. Load `day1/generated/retail_panel_issues.csv`, drop rows where `sales` is missing, sort by date.
> 2. Use a **chronological split** (first 70% by date = train, rest = test).
> 3. Compare four `LinearRegression` models with different feature sets, reporting **train R²**, **test R²**, and **gap**.
> 4. Decide what this tells you about raw IDs, grouped features, and date features.
>
> Follow these steps:
>
> **Step 1 — Load and split chronologically**: sort by date, take the first 70% as train. This is different from `train_test_split` — here order matters.
>
> **Step 2 — Define base features**: start with `["inventory_units", "promotion_discount_pct", "store_traffic_index"]`.
>
> **Step 3 — Add features incrementally** and fit a `Pipeline([("imputer", SimpleImputer(...)), ("model", LinearRegression())])` for each variant:
> - **(a)** base features only
> - **(b)** base + raw `store_id`
> - **(c)** base + `store_mean_sales_train` (compute `groupby("store_id")["sales"].mean()` on **train only**, then `.map()` onto both splits — same pattern as the group aggregates in the lesson notebook)
> - **(d)** base + store mean + date features (`dt.month`, `dt.dayofweek` — same pattern as the datetime cell in the lesson notebook)
>
> **Step 4 — Score with `r2_score`** on both train and test for each variant.
>
> *Hint: if you compute a store-level sales aggregate, learn it on the training period only.*

In [ ]:
# Step 1: Load and chronological split
# retail = pd.read_csv("day1/generated/retail_panel_issues.csv")
# retail = retail[retail["sales"].notna()].copy()
# retail["date"] = pd.to_datetime(retail["date"])
# retail = retail.sort_values("date").reset_index(drop=True)
# split_idx = int(len(retail) * 0.7)
# retail_train = retail.iloc[:split_idx].copy()
# retail_test = retail.iloc[split_idx:].copy()

# Step 2: Base features
# base_cols = ["inventory_units", "promotion_discount_pct", "store_traffic_index"]

# Step 3a: Group aggregate (train only, then map — same pattern as lesson notebook)
# store_mean_sales = retail_train.groupby("store_id")["sales"].mean()
# global_mean = retail_train["sales"].mean()
# for df in [retail_train, retail_test]:
#     df["store_mean_sales_train"] = df["store_id"].map(store_mean_sales).fillna(global_mean)

# Step 3b: Date features (same pattern as lesson notebook)
# for df in [retail_train, retail_test]:
#     df["sale_month"] = df["date"].dt.month
#     df["sale_dayofweek"] = df["date"].dt.dayofweek

# Step 4: For each feature set, fit Pipeline and score with r2_score
# pipe = Pipeline([
#     ("imputer", SimpleImputer(strategy="median")),
#     ("model", LinearRegression()),
# ])
# pipe.fit(retail_train[feature_cols], retail_train["sales"])
# train_r2 = r2_score(retail_train["sales"], pipe.predict(retail_train[feature_cols]))
# test_r2 = r2_score(retail_test["sales"], pipe.predict(retail_test[feature_cols]))

### Solution / Considerations

One possible path is below.


In [ ]:
# ── Step 1: Load and chronological split ──
retail = pd.read_csv("day1/generated/retail_panel_issues.csv")
retail = retail[retail["sales"].notna()].copy()
retail["date"] = pd.to_datetime(retail["date"])
retail = retail.sort_values("date").reset_index(drop=True)

split_idx = int(len(retail) * 0.7)
retail_train = retail.iloc[:split_idx].copy()
retail_test = retail.iloc[split_idx:].copy()
print(f"Chronological split: {retail_train.shape[0]:,} train rows | {retail_test.shape[0]:,} test rows")

# ── Step 2: Base features ──
base_cols = ["inventory_units", "promotion_discount_pct", "store_traffic_index"]

# ── Step 3a: Group aggregate (train only) ──
store_mean_sales = retail_train.groupby("store_id")["sales"].mean()
store_global_mean = retail_train["sales"].mean()
for df in [retail_train, retail_test]:
    df["store_mean_sales_train"] = df["store_id"].map(store_mean_sales).fillna(store_global_mean)

# ── Step 3b: Date features ──
for df in [retail_train, retail_test]:
    df["sale_month"] = df["date"].dt.month
    df["sale_dayofweek"] = df["date"].dt.dayofweek
    df["sale_week"] = df["date"].dt.isocalendar().week.astype(int)

# ── Step 4: Fit and score each variant ──
variants = [
    ("Base features", base_cols),
    ("Base + raw store_id", base_cols + ["store_id"]),
    ("Base + store_mean_sales_train", base_cols + ["store_mean_sales_train"]),
    ("Base + store mean + date features", base_cols + ["store_mean_sales_train", "sale_month", "sale_dayofweek", "sale_week"]),
]

results = []
for name, feature_cols in variants:
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LinearRegression()),
    ])
    pipe.fit(retail_train[feature_cols], retail_train["sales"])

    train_r2 = r2_score(retail_train["sales"], pipe.predict(retail_train[feature_cols]))
    test_r2 = r2_score(retail_test["sales"], pipe.predict(retail_test[feature_cols]))
    results.append({"Model": name, "Train R²": train_r2, "Test R²": test_r2, "Gap": train_r2 - test_r2})

retail_comparison = pd.DataFrame(results)
retail_comparison["Test R² gain vs base"] = retail_comparison["Test R²"] - retail_comparison.loc[0, "Test R²"]
display(retail_comparison)

**Considerations**

- Treating `store_id` as a raw numeric feature usually adds almost nothing: the model sees an arbitrary code, not context.
- A **train-only group aggregate** such as `store_mean_sales_train` is much more useful because it encodes something the model can actually use.
- Simple datetime features such as month, week, or day of week can add extra signal on top of the group context.
- This is close to the ID lesson: raw identifiers are often useless or dangerous, but repeated group identifiers can become useful once you convert them into contextual features.


---

## <a id="capstone"></a> Exercise 12 — Capstone: From Raw Data to a Usable Feature Matrix

This final exercise shifts from isolated pitfalls to an end-to-end preprocessing workflow.

You will work on one **local dataset**: `day1/generated/king_county_capstone.csv`.

Your goal is to turn the raw table into a **numeric, leakage-safe feature matrix suitable for a scaling-sensitive model** such as a regularized linear model.

That means:
- no target or non-feature columns inside the final matrix
- no raw string/object columns left
- no accidental use of `val` or `test` to fit imputers, encoders, or scalers
- a documented policy for missing values, categoricals, invalid values, date features, grouped features, and scaling

Date-derived and grouped features are encouraged here.

In [ ]:
# ── Load capstone dataset ──
capstone = pd.read_csv("day1/generated/king_county_capstone.csv", parse_dates=["date"])

print("King County capstone")
print(f"  Shape: {capstone.shape}")
print(f"  Splits: {capstone['split'].value_counts().sort_index().to_dict()}")
print(f"  Target: 'price' — median: ${capstone['price'].median():,.0f}")
print()

missing_summary = (
    capstone.isna().mean().mul(100).rename("missing_pct").sort_values(ascending=False).to_frame()
)
print("Columns with missing values:")
display(missing_summary.query("missing_pct > 0"))
print()
print("Dtypes:")
display(capstone.dtypes.rename("dtype").to_frame())
print()
show(capstone)

> **Capstone Task**
>
> Starting from the raw dataframe above, produce:
>
> 1. A preprocessing policy table with columns such as `column_or_group`, `issue`, `choice`, `why`.
> 2. A clean train / val / test split that uses the provided `split` column.
> 3. Final feature matrices named `X_train_final`, `X_val_final`, `X_test_final`.
> 4. Matching targets named `y_train`, `y_val`, `y_test`.
> 5. A short log of engineered features you created from dates or grouped relationships.
>
> Constraints:
> - Assume the downstream model is **scale-sensitive**, so your final matrix should be appropriate for a linear model.
> - Fit every learned step on `train` only.
> - Do not use `price`, `split`, IDs, or process metadata as features.
> - Handle unseen categories safely.
> - End with a matrix that is numeric and has no unintended missing values.

In [ ]:
# Your capstone work starts here

# Suggested variable names:
# train_cap = capstone.loc[capstone["split"] == "train"].copy()
# val_cap = capstone.loc[capstone["split"] == "val"].copy()
# test_cap = capstone.loc[capstone["split"] == "test"].copy()
# y_train = train_cap["price"].copy()
# y_val = val_cap["price"].copy()
# y_test = test_cap["price"].copy()
# X_train_final = ...
# X_val_final = ...
# X_test_final = ...

In [ ]:
# Optional self-checks after you build the final matrices

# assert list(X_train_final.columns) == list(X_val_final.columns) == list(X_test_final.columns)
# assert X_train_final.select_dtypes(exclude=[np.number]).shape[1] == 0
# assert X_val_final.select_dtypes(exclude=[np.number]).shape[1] == 0
# assert X_test_final.select_dtypes(exclude=[np.number]).shape[1] == 0
# assert not X_train_final.isna().any().any()
# assert not X_val_final.isna().any().any()
# assert not X_test_final.isna().any().any()
# print("Capstone matrices look model-ready.")

---

## Scratch Space

In [ ]:
# Free exploration
